In [2]:
import numpy as np
from moabb.datasets import BNCI2014_001
from moabb.paradigms import MotorImagery, LeftRightImagery
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from mne.decoding import CSP
from moabb.evaluations import CrossSubjectEvaluation
from sklearn.pipeline import make_pipeline
from scipy import signal
from scipy.io import loadmat
import os
import mne
from scipy.linalg import logm, expm
from sklearn.svm import SVC
from sklearn.metrics import balanced_accuracy_score

In [3]:
active_all_event_ids = {'769': 769, '770': 770, '771': 771, '772': 772}
active_lr_event_ids = {'769': 769, '770': 770}
unknown_event_id = {'783': 783} 

In [4]:
data_dir = '/home/vishwa/eeg_tl/Recreating papers/BCICIV_2a'

In [5]:
# Define a causal bandpass filter function using a Butterworth design.
def causal_bandpass_filter(data, lowcut=8, highcut=30, fs=250, order=5):
    nyq = 0.5 * fs
    # Normalize the cutoff frequencies (Matlab's fir1 expects normalized cutoff frequencies
    low = lowcut / nyq
    high = highcut / nyq
    # Design the FIR filter. Note: order+1 coefficients are returned to match Matlab's fir1 which returns n+1 taps.
    b = signal.firwin(order + 1, [low, high], window='hamming', pass_zero=False)
    # Apply the filter causally using lfilter (this introduces a constant delay).
    filtered_data = signal.lfilter(b, [1.0], data)
    return filtered_data

In [6]:
# With a sampling frequency of 250 Hz, 1001 samples equate to 1001/250 seconds.
sfreq = 250
tmin = 0       # Epoch start at cue onset.
# Set tmax so that n_samples = (tmax-tmin)*sfreq + 1 = 1001, i.e. 4 seconds long.
tmax = (1001 - 1) / sfreq  # This gives 4.0 seconds.

In [7]:
train_active_X = []         # List to hold numpy arrays with shape (n_trials, 22, 1001) per subject.
train_active_y = []         # List to hold event labels per subject.
train_active_metadata = []  # List to hold event metadata per subject.

# Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
for subj in range(1, 10):
    filename = os.path.join(data_dir, f'A{subj:02d}T.gdf')
    
    # Read the GDF file (using preload=True to load data into memory).
    train_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
    # Retain only EEG channels (22 channels) and exclude EOG channels.
    train_raw.pick_types(eeg=True, eog=False)
    
    # Extract events corresponding only to the four desired types.
    train_active_events, _ = mne.events_from_annotations(train_raw, event_id=active_lr_event_ids)
    
    # Create epochs from tmin to tmax.
    train_active_epochs = mne.Epochs(train_raw, train_active_events, event_id=active_lr_event_ids, tmin=tmin, tmax=tmax,
                        baseline=None, preload=True, verbose=False)
    
    # Get the epoch data (num_epochs x 22 channels x 1001 samples).
    train_active_data = train_active_epochs.get_data()
    
    # Print the number of extracted epochs to verify
    print(f"Subject {subj}: Epoch data shape {train_active_data.shape}")
    
    # Sampling frequency from raw.info (should be 250).
    fs = int(train_raw.info['sfreq'])
    # print(fs)
    n_trials, n_channels, n_times = train_active_data.shape
    train_active_filtered_data = np.empty_like(train_active_data)
    
    # Apply the causal bandpass filter channel‐wise for each trial.
    for trial in range(n_trials):
        for ch in range(n_channels):
            train_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                train_active_data[trial, ch, :],
                lowcut=8,    # Lower bound of sensorimotor rhythm.
                highcut=30,  # Upper bound of sensorimotor rhythm.
                fs=fs,
                order=50     # Lower order for a smoother causal filter.
            )
    
    # Append the processed data, labels, and event metadata.
    train_active_X.append(train_active_filtered_data)
    train_active_y.append(train_active_epochs.events[:, 2])  # The third column holds the event code.
    train_active_metadata.append(train_active_epochs.events)

print("Loaded data for", len(train_active_X), "subjects.")

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 1: Epoch data shape (144, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 2: Epoch data shape (144, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 3: Epoch data shape (144, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 4: Epoch data shape (144, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 5: Epoch data shape (144, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['769', '770']
Subject 6: Epoch data shape (144, 22, 1001

In [8]:
eval_active_X = []         
eval_active_y = []        
eval_active_metadata = [] 

# Loop over subjects. Assume files are named "A01T.gdf", "A02T.gdf", ..., "A09T.gdf".
for subj in range(1, 10):
    filename = os.path.join(data_dir, f'A{subj:02d}E.gdf')
    mat_data = loadmat(f'/home/vishwa/eeg_tl/Recreating papers/BCICIV_2A true labels/A{subj:02d}E.mat')
    true_y =  np.array(mat_data['classlabel'], dtype=np.int64).reshape(288,) + 768
    # Read the GDF file (using preload=True to load data into memory).
    eval_raw = mne.io.read_raw_gdf(filename, preload=True, verbose=False, eog=['EOG-left', 'EOG-central', 'EOG-right'])
    # Retain only EEG channels (22 channels) and exclude EOG channels.
    eval_raw.pick_types(eeg=True, eog=False)
    
    # Extract events corresponding only to the four desired types.
    eval_active_events, _ = mne.events_from_annotations(eval_raw, event_id=unknown_event_id)
    
    # Create epochs from tmin to tmax.
    eval_active_epochs = mne.Epochs(eval_raw, eval_active_events, event_id=unknown_event_id, tmin=tmin, tmax=tmax,
                        baseline=None, preload=True, verbose=False)
    
    # Get the epoch data (num_epochs x 22 channels x 1001 samples).
    eval_active_data = eval_active_epochs.get_data()
    
    # Print the number of extracted epochs to verify
    print(f"Subject {subj}: Epoch data shape {eval_active_data.shape}")
    
    # Sampling frequency from raw.info (should be 250).
    fs = int(eval_raw.info['sfreq'])
    # print(fs)
    n_trials, n_channels, n_times = eval_active_data.shape
    eval_active_filtered_data = np.empty_like(eval_active_data)
    
    # Apply the causal bandpass filter channel‐wise for each trial.
    for trial in range(n_trials):
        for ch in range(n_channels):
            eval_active_filtered_data[trial, ch, :] = causal_bandpass_filter(
                eval_active_data[trial, ch, :],
                lowcut=8,    # Lower bound of sensorimotor rhythm.
                highcut=30,  # Upper bound of sensorimotor rhythm.
                fs=fs,
                order=50     # Lower order for a smoother causal filter.
            )
    
    # Append the processed data, labels, and event metadata.
    eval_active_X.append(eval_active_filtered_data)
    eval_active_y.append(true_y)  # The third column holds the event code.
    eval_active_metadata.append(eval_active_epochs.events)

print("Loaded data for", len(eval_active_X), "subjects.")

NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 1: Epoch data shape (288, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 2: Epoch data shape (288, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 3: Epoch data shape (288, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 4: Epoch data shape (288, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 5: Epoch data shape (288, 22, 1001)
NOTE: pick_types() is a legacy function. New code should use inst.pick(...).
Used Annotations descriptions: ['783']
Subject 6: Epoch data shape (288, 22, 1001)
NOTE: pick_types() is a legacy function.

In [9]:
eval_active_X = [x[np.isin(y, [769, 770])] for x, y in zip(eval_active_X, eval_active_y)]
eval_active_y = [y[np.isin(y, [769, 770])] for y in eval_active_y]

In [10]:
import numpy as np
from pyriemann.utils.base import logm, expm, sqrtm, invsqrtm
from pyriemann.utils.mean import mean_riemann
from pyriemann.utils.distance import distance_riemann
from pyriemann.classification import MDM
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from pyriemann.tangentspace import TangentSpace
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

# Print data shapes - we'll only use session 1 data (train_active_X)
print(f"Number of subjects: {len(train_active_X)}")
print(f"Training data shape for first subject: {train_active_X[0].shape}")
print(f"Unique labels in training: {np.unique(train_active_y[0])}")

# Helper function for better covariance estimation
def compute_scm(X):
    """Compute sample covariance matrix with proper regularization"""
    n_trials, n_channels, n_times = X.shape
    covs = np.zeros((n_trials, n_channels, n_channels))
    
    for trial in range(n_trials):
        # Center the data
        trial_data = X[trial]
        trial_data = trial_data - trial_data.mean(axis=1, keepdims=True)
        
        # Spatial covariance: X*X^T/(n_times-1)
        covs[trial] = (trial_data @ trial_data.T) / (n_times - 1)
        
        # Add small regularization
        covs[trial] = (1 - 0.05) * covs[trial] + 0.05 * np.eye(n_channels) * np.trace(covs[trial]) / n_channels
    
    return covs

# Compute spatial covariance matrices with improved estimation
# We only use session 1 data
all_covs = []
all_labels = []

for subj_idx in range(len(train_active_X)):
    X = train_active_X[subj_idx]
    y = train_active_y[subj_idx]
    
    covs = compute_scm(X)
    all_covs.append(covs)
    
    # Normalize labels from 769/770 to 0/1
    labels_norm = (y == 770).astype(int)
    all_labels.append(labels_norm)

# Labeled trials counts as used in the paper
labeled_trials_counts = [1, 6, 18, 36]
all_results = {}

n_subjects = len(all_covs)

print("Running cross-subject transfer learning experiments...")

for n_labeled in labeled_trials_counts:
    print(f"\nEvaluating with {n_labeled} labeled trials from target")
    
    # Matrices to store results
    cross_subject_acc = np.zeros((n_subjects, n_subjects))
    cross_subject_auc = np.zeros((n_subjects, n_subjects))
    baseline_acc = np.zeros((n_subjects, n_subjects))
    baseline_auc = np.zeros((n_subjects, n_subjects))
    rct_acc = np.zeros((n_subjects, n_subjects))
    rct_auc = np.zeros((n_subjects, n_subjects))
    
    for target_idx in range(n_subjects):
        target_covs = all_covs[target_idx]
        target_labels = all_labels[target_idx]
        
        # To make results more robust, we will use cross-validation with multiple data splits
        # This averages out the effect of specific labeled samples
        n_repeats = 5  # Repeat with different random seeds
        cv_baseline_aucs = []
        cv_rct_aucs = []
        cv_rpa_aucs = []
        
        for seed in range(n_repeats):
            # Split target data into labeled and unlabeled
            np.random.seed(seed)
            
            # Ensure balanced class representation
            unique_classes = np.unique(target_labels)
            labeled_indices = []
            
            for c in unique_classes:
                class_indices = np.where(target_labels == c)[0]
                np.random.shuffle(class_indices)  # Shuffle to get different subsets
                samples_per_class = max(1, n_labeled // len(unique_classes))
                if len(class_indices) >= samples_per_class:
                    labeled_indices.extend(class_indices[:samples_per_class])
            
            # Ensure exact number of labeled trials
            labeled_indices = labeled_indices[:min(n_labeled, len(target_labels))]
            unlabeled_indices = [i for i in range(len(target_labels)) if i not in labeled_indices]
            
            # Skip if we don't have enough samples in any split
            if len(labeled_indices) < 2 or len(unlabeled_indices) < 2:
                continue
            
            # Split target data
            target_labeled_covs = target_covs[labeled_indices]
            target_labeled_labels = target_labels[labeled_indices]
            target_unlabeled_covs = target_covs[unlabeled_indices]
            target_unlabeled_labels = target_labels[unlabeled_indices]
            
            for source_idx in range(n_subjects):
                if source_idx == target_idx:
                    # Skip if source and target are the same subject
                    continue
                
                source_covs = all_covs[source_idx]
                source_labels = all_labels[source_idx]
                
                # 1. Baseline: Direct transfer (DCT)
                try:
                    # For BCI data, MDM with Riemannian distance is a good baseline
                    mdm_baseline = MDM(metric='riemann')
                    mdm_baseline.fit(source_covs, source_labels)
                    
                    baseline_preds = mdm_baseline.predict(target_unlabeled_covs)
                    baseline_proba = mdm_baseline.predict_proba(target_unlabeled_covs)[:, 1]
                    
                    # Store results for this fold
                    baseline_acc[target_idx, source_idx] += accuracy_score(target_unlabeled_labels, baseline_preds) / n_repeats
                    baseline_auc_score = roc_auc_score(target_unlabeled_labels, baseline_proba)
                    baseline_auc[target_idx, source_idx] += baseline_auc_score / n_repeats
                    cv_baseline_aucs.append(baseline_auc_score)
                except:
                    pass
                
                # 2. RCT: Recentering method
                try:
                    # Compute geometric means
                    M_source = mean_riemann(source_covs)
                    M_target = mean_riemann(target_labeled_covs)
                    
                    # Re-center source matrices
                    source_recentered = np.zeros_like(source_covs)
                    M_source_invsqrt = invsqrtm(M_source)
                    
                    for i in range(len(source_covs)):
                        source_recentered[i] = M_source_invsqrt @ source_covs[i] @ M_source_invsqrt
                    
                    # Re-center target matrices
                    target_labeled_recentered = np.zeros_like(target_labeled_covs)
                    target_unlabeled_recentered = np.zeros_like(target_unlabeled_covs)
                    M_target_invsqrt = invsqrtm(M_target)
                    
                    for i in range(len(target_labeled_covs)):
                        target_labeled_recentered[i] = M_target_invsqrt @ target_labeled_covs[i] @ M_target_invsqrt
                    
                    for i in range(len(target_unlabeled_covs)):
                        target_unlabeled_recentered[i] = M_target_invsqrt @ target_unlabeled_covs[i] @ M_target_invsqrt
                    
                    # Train classifier on recentered data
                    # Paper likely used Tangent Space projection + classifier
                    # This generally gives higher AUC than just MDM
                    ts_clf_rct = make_pipeline(
                        TangentSpace(metric='riemann'),
                        LogisticRegression(C=1.0, solver='liblinear')
                    )
                    
                    # Combine source and labeled target data
                    train_data_rct = np.vstack((source_recentered, target_labeled_recentered))
                    train_labels_rct = np.concatenate((source_labels, target_labeled_labels))
                    
                    ts_clf_rct.fit(train_data_rct, train_labels_rct)
                    
                    rct_preds = ts_clf_rct.predict(target_unlabeled_recentered)
                    rct_proba = ts_clf_rct.predict_proba(target_unlabeled_recentered)[:, 1]
                    
                    rct_acc[target_idx, source_idx] += accuracy_score(target_unlabeled_labels, rct_preds) / n_repeats
                    rct_auc_score = roc_auc_score(target_unlabeled_labels, rct_proba)
                    rct_auc[target_idx, source_idx] += rct_auc_score / n_repeats
                    cv_rct_aucs.append(rct_auc_score)
                except:
                    pass
                
                # 3. RPA: Full Riemannian Procrustes Analysis
                try:
                    # Step 1: Already done in RCT (recentering)
                    
                    # Step 2: Equalize dispersions
                    d_source = 0
                    for cov in source_covs:
                        d_source += distance_riemann(M_source, cov) ** 2
                    d_source /= len(source_covs)
                    
                    d_target = 0
                    for cov in target_labeled_covs:
                        d_target += distance_riemann(M_target, cov) ** 2
                    d_target /= max(1, len(target_labeled_covs))
                    
                    # Avoid division by zero
                    if d_target < 1e-10:
                        d_target = 1e-10
                        
                    scaling_factor = np.sqrt(d_source / d_target)
                    
                    # Scale target matrices
                    target_labeled_scaled = np.zeros_like(target_labeled_recentered)
                    for i in range(len(target_labeled_recentered)):
                        log_cov = logm(target_labeled_recentered[i])
                        target_labeled_scaled[i] = expm(log_cov / scaling_factor)
                    
                    # Step 3: Compute orthogonal matrix for rotation
                    # Compute class means for both datasets
                    source_class_means = {}
                    target_class_means = {}
                    
                    for c in unique_classes:
                        source_indices = np.where(source_labels == c)[0]
                        target_indices = np.where(target_labeled_labels == c)[0]
                        
                        if len(source_indices) > 0 and len(target_indices) > 0:
                            source_class_means[c] = mean_riemann(source_recentered[source_indices])
                            target_class_means[c] = mean_riemann(target_labeled_scaled[target_indices])
                    
                    # Compute G_k and G_k_tilde matrices
                    G_k = {}
                    G_k_tilde = {}
                    
                    for c in source_class_means:
                        if c in target_class_means:
                            G_k[c] = M_source_invsqrt @ source_class_means[c] @ M_source_invsqrt
                            G_k_tilde[c] = M_target_invsqrt @ target_class_means[c] @ M_target_invsqrt
                    
                    # Find orthogonal matrix U using eigendecomposition
                    Q_k = {}
                    Q_k_tilde = {}
                    
                    for c in G_k:
                        eigvals, eigvecs = np.linalg.eigh(G_k[c])
                        idx = eigvals.argsort()[::-1]
                        Q_k[c] = eigvecs[:, idx]
                        
                        eigvals_tilde, eigvecs_tilde = np.linalg.eigh(G_k_tilde[c])
                        idx_tilde = eigvals_tilde.argsort()[::-1]
                        Q_k_tilde[c] = eigvecs_tilde[:, idx_tilde]
                    
                    # Compute U as weighted average of Q_k_tilde * Q_k^T
                    n_channels = source_covs.shape[1]
                    U = np.zeros((n_channels, n_channels))
                    total_weight = 0
                    
                    for c in Q_k:
                        weight = len(np.where(source_labels == c)[0])
                        U += weight * Q_k_tilde[c] @ Q_k[c].T
                        total_weight += weight
                    
                    if total_weight > 0:
                        U /= total_weight
                    
                    # Ensure U is orthogonal using SVD
                    u, s, vh = np.linalg.svd(U, full_matrices=True)
                    U = u @ vh
                    
                    # Apply rotation to target matrices
                    target_labeled_rotated = np.zeros_like(target_labeled_scaled)
                    for i in range(len(target_labeled_scaled)):
                        target_labeled_rotated[i] = U.T @ target_labeled_scaled[i] @ U
                    
                    # Apply full transformation pipeline to unlabeled target data
                    target_unlabeled_transformed = np.zeros_like(target_unlabeled_covs)
                    for i in range(len(target_unlabeled_covs)):
                        # Re-center
                        recentered = M_target_invsqrt @ target_unlabeled_covs[i] @ M_target_invsqrt
                        
                        # Scale
                        log_cov = logm(recentered)
                        scaled = expm(log_cov / scaling_factor)
                        
                        # Rotate
                        target_unlabeled_transformed[i] = U.T @ scaled @ U
                    
                    # Train classifier on transformed data
                    # Use Tangent Space + LogisticRegression
                    ts_clf_rpa = make_pipeline(
                        TangentSpace(metric='riemann'),
                        LogisticRegression(C=1.0, solver='liblinear')
                    )
                    
                    # Combine source and labeled target data
                    train_data_rpa = np.vstack((source_recentered, target_labeled_rotated))
                    train_labels_rpa = np.concatenate((source_labels, target_labeled_labels))
                    
                    ts_clf_rpa.fit(train_data_rpa, train_labels_rpa)
                    
                    rpa_preds = ts_clf_rpa.predict(target_unlabeled_transformed)
                    rpa_proba = ts_clf_rpa.predict_proba(target_unlabeled_transformed)[:, 1]
                    
                    rpa_acc = accuracy_score(target_unlabeled_labels, rpa_preds)
                    rpa_auc = roc_auc_score(target_unlabeled_labels, rpa_proba)
                    
                    cross_subject_acc[target_idx, source_idx] += rpa_acc / n_repeats
                    cross_subject_auc[target_idx, source_idx] += rpa_auc / n_repeats
                    cv_rpa_aucs.append(rpa_auc)
                except Exception as e:
                    pass
        
        # Print average results for this target subject
        avg_baseline = np.mean([x for x in cv_baseline_aucs if x > 0]) if cv_baseline_aucs else 0
        avg_rct = np.mean([x for x in cv_rct_aucs if x > 0]) if cv_rct_aucs else 0
        avg_rpa = np.mean([x for x in cv_rpa_aucs if x > 0]) if cv_rpa_aucs else 0
        
        print(f"\nTarget Subject {target_idx+1} - Average AUC across sources:")
        print(f"  Baseline (DCT): {avg_baseline:.4f}")
        print(f"  RCT:            {avg_rct:.4f}")
        print(f"  RPA:            {avg_rpa:.4f}")
    
    # Store results for this number of labeled trials
    all_results[n_labeled] = {
        'baseline_acc': baseline_acc.copy(),
        'baseline_auc': baseline_auc.copy(),
        'rct_acc': rct_acc.copy(),
        'rct_auc': rct_auc.copy(),
        'rpa_acc': cross_subject_acc.copy(),
        'rpa_auc': cross_subject_auc.copy()
    }
    
    # Calculate average performance (excluding diagonal)
    mask = ~np.eye(n_subjects, dtype=bool)
    avg_baseline_acc = np.mean([x for x in baseline_acc[mask].flatten() if x > 0])
    avg_baseline_auc = np.mean([x for x in baseline_auc[mask].flatten() if x > 0])
    avg_rct_acc = np.mean([x for x in rct_acc[mask].flatten() if x > 0])
    avg_rct_auc = np.mean([x for x in rct_auc[mask].flatten() if x > 0])
    avg_rpa_acc = np.mean([x for x in cross_subject_acc[mask].flatten() if x > 0])
    avg_rpa_auc = np.mean([x for x in cross_subject_auc[mask].flatten() if x > 0])
    
    print(f"\nOverall Results with {n_labeled} labeled trials:")
    print(f"Baseline (DCT): ACC={avg_baseline_acc:.4f}, AUC={avg_baseline_auc:.4f}")
    print(f"RCT:            ACC={avg_rct_acc:.4f}, AUC={avg_rct_auc:.4f}")
    print(f"RPA:            ACC={avg_rpa_acc:.4f}, AUC={avg_rpa_auc:.4f}")

# Visualize the results in a format matching the paper
labeled_counts = list(all_results.keys())
avg_baseline_auc = []
avg_rct_auc = []
avg_rpa_auc = []

for n in labeled_counts:
    mask = ~np.eye(n_subjects, dtype=bool)
    avg_baseline_auc.append(np.mean([x for x in all_results[n]['baseline_auc'][mask].flatten() if x > 0]))
    avg_rct_auc.append(np.mean([x for x in all_results[n]['rct_auc'][mask].flatten() if x > 0]))
    avg_rpa_auc.append(np.mean([x for x in all_results[n]['rpa_auc'][mask].flatten() if x > 0]))

plt.figure(figsize=(10, 6))
plt.plot(labeled_counts, avg_baseline_auc, 'o-', color='#FF5555', label='DCT (Direct Transfer)')
plt.plot(labeled_counts, avg_rct_auc, 's-', color='#8B4513', label='RCT (Recentering)')
plt.plot(labeled_counts, avg_rpa_auc, 'd-', color='#228B22', label='RPA')
plt.xlabel('Number of Labeled Target Trials')
plt.ylabel('Average AUC')
plt.title('Cross-Subject Transfer Performance on BCI IV 2a Dataset')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('rpa_cross_subject_results.png')
plt.show()

# Table of results matching the paper's format
print("\nCross-Subject AUC Results (format matching paper Table II):")
print("N  | DCT  | RCT  | RPA")
print("-" * 25)
for i, n in enumerate(labeled_counts):
    print(f"{n:2d} | {avg_baseline_auc[i]:.2f} | {avg_rct_auc[i]:.2f} | {avg_rpa_auc[i]:.2f}")

Number of subjects: 9
Training data shape for first subject: (144, 22, 1001)
Unique labels in training: [769 770]
Running cross-subject transfer learning experiments...

Evaluating with 1 labeled trials from target

Target Subject 1 - Average AUC across sources:
  Baseline (DCT): 0.0000
  RCT:            0.0000
  RPA:            0.0000

Target Subject 2 - Average AUC across sources:
  Baseline (DCT): 0.0000
  RCT:            0.0000
  RPA:            0.0000

Target Subject 3 - Average AUC across sources:
  Baseline (DCT): 0.0000
  RCT:            0.0000
  RPA:            0.0000

Target Subject 4 - Average AUC across sources:
  Baseline (DCT): 0.0000
  RCT:            0.0000
  RPA:            0.0000

Target Subject 5 - Average AUC across sources:
  Baseline (DCT): 0.0000
  RCT:            0.0000
  RPA:            0.0000

Target Subject 6 - Average AUC across sources:
  Baseline (DCT): 0.0000
  RCT:            0.0000
  RPA:            0.0000

Target Subject 7 - Average AUC across sources:

KeyboardInterrupt: 